<a href="https://colab.research.google.com/github/veneya/redrob-hackathon/blob/main/korean_hacakthon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import math

SKILL_ALIASES = {
    "python": "python", "pyhton": "python",
    "java": "java",
    "javascript": "javascript", "javascrpit": "javascript", "js": "javascript",
    "typescript": "typescript", "typescrpit": "typescript",
    "c++": "cpp", "cpp": "cpp",
    "r": "r",
    "kotlin": "kotlin",
    "machinelearning": "machine_learning", "machine learning": "machine_learning",
    "ml": "machine_learning", "sklearn": "machine_learning",
    "deeplearning": "deep_learning", "deep learning": "deep_learning",
    "deep-learning": "deep_learning",
    "tensorflow": "tensorflow", "pytorch": "pytorch", "keras": "keras",
    "nlp": "nlp", "bert": "bert", "xgboost": "xgboost",
    "feature engineering": "feature_engineering",
    "statistics": "statistics", "stats": "statistics",
    "regression": "regression", "clustering": "clustering",
    "data-viz": "data_visualization", "data visualization": "data_visualization",
    "data viz": "data_visualization", "matplotlib": "data_visualization",
    "tableau": "data_visualization", "power-bi": "data_visualization",
    "power bi": "data_visualization", "powerbi": "data_visualization",
    "pandas": "pandas", "numpy": "numpy",
    "react": "react", "reacts": "react", "reactjs": "react",
    "vue": "vue", "vue.js": "vue", "vuejs": "vue",
    "redux": "redux", "tailwind": "tailwind",
    "html/css": "html_css", "html css": "html_css",
    "html": "html_css", "css": "html_css",
    "jest": "jest", "graphql": "graphql",
    "node.js": "nodejs", "nodejs": "nodejs", "node js": "nodejs",
    "flask": "flask",
    "spring boot": "spring_boot", "springboot": "spring_boot",
    "rest api": "rest_api", "rest": "rest_api", "restapi": "rest_api",
    "microservices": "microservices",
    "sql": "sql", "mysql": "mysql", "mysq": "mysql",
    "postgresql": "postgresql", "postgres": "postgresql",
    "mongodb": "mongodb", "redis": "redis",
    "docker": "docker",
    "kubernetes": "kubernetes", "kubernates": "kubernetes", "k8s": "kubernetes",
    "ci/cd": "ci_cd", "cicd": "ci_cd", "ci cd": "ci_cd",
    "aws": "aws",
    "android": "android", "firebase": "firebase",
    "algorithms": "algorithms", "algoritms": "algorithms",
    "data structure": "data_structures", "data structures": "data_structures",
    "competitive programming": "competitive_programming",
    "ui/ux": "ui_ux", "ui ux": "ui_ux", "figma": "figma",
}

In [11]:
resumes_raw = {
    "Arjun Sharma":    "Pyhton, MachineLearning, SQL, pandas, numpy, Deep-learning",
    "Priya Nair":      "JavaScrpit, Reacts, Node.JS, MongoDb, REST api, HTML/CSS",
    "Rahul Gupta":     "Java, Spring Boot, MySql, Microservices, Docker, kubernates",
    "Sneha Patel":     "Python, TensorFlow, Keras, NLP, BERT, data-viz, matplotlib",
    "Vikram Singh":    "C++, Algoritms, Data Structure, competitive programming, python",
    "Ananya Krishnan": "javascript, vue.js, python, flask, PostgreSQL, AWS, CI/CD",
    "Karan Mehta":     "Python, Sklearn, XGboost, feature engineering, SQL, tableau",
    "Deepika Rao":     "Java, Android, Kotlin, Firebase, REST, UI/UX, figma",
    "Aditya Kumar":    "Reactjs, TypeScrpit, GraphQL, redux, tailwind, nodejs, jest",
    "Meera Iyer":      "python, R, statistics, ML, regression, clustering, Power-BI",
}

job_descriptions = {
    "JD-1": {
        "company": "Kakao", "role": "ML Engineer",
        "skills": "Python, Machine Learning, Deep Learning, TensorFlow, PyTorch, SQL, Data Visualization, NLP, BERT, Feature Engineering, Statistics",
    },
    "JD-2": {
        "company": "Naver", "role": "Backend Engineer",
        "skills": "Java, Spring Boot, MySQL, PostgreSQL, Microservices, Docker, Kubernetes, REST API, CI/CD, Redis",
    },
    "JD-3": {
        "company": "Line", "role": "Frontend Engineer",
        "skills": "JavaScript, React, Vue, TypeScript, REST API, HTML/CSS, Node.js, GraphQL, Redux, Jest, AWS",
    },
}

In [12]:
def normalize_skills(skill_string):
    seen = set()
    result = []
    for skill in skill_string.split(','):      # Step 1: split on comma
        token = skill.strip().lower()          # Step 2: strip + lowercase
        canonical = SKILL_ALIASES.get(token)   # Step 3: alias lookup
        if canonical and canonical not in seen: # Step 4: discard unknown + deduplicate
            seen.add(canonical)
            result.append(canonical)
    return result

# Apply to all resumes
resumes_normalized = {
    name: normalize_skills(raw_string)
    for name, raw_string in resumes_raw.items()
}

In [13]:
# Shared vocabulary from resume skills only, sorted A→Z
all_skills = set()
for skills in resumes_normalized.values():
    all_skills.update(skills)
vocabulary = sorted(all_skills)

# Document frequency: count how many resumes contain each skill
df = {skill: 0 for skill in vocabulary}
for skills in resumes_normalized.values():
    for skill in skills:
        df[skill] += 1

# TF = 1/N,  IDF = ln(10/df),  TF-IDF = TF x IDF
def compute_tfidf(name, skills):
    n = len(skills)
    vector = {}
    for skill in vocabulary:
        if skill in skills:
            tf  = 1 / n
            idf = math.log(10 / df[skill])
            vector[skill] = tf * idf
        else:
            vector[skill] = 0.0
    return vector

tfidf_vectors = {
    name: compute_tfidf(name, skills)
    for name, skills in resumes_normalized.items()
}

In [14]:
# Binary vector: 1 if JD needs the skill, 0 otherwise
def build_jd_vector(jd_skill_string):
    normalized_jd = set(normalize_skills(jd_skill_string))
    return {skill: (1 if skill in normalized_jd else 0) for skill in vocabulary}

jd_vectors = {
    jd_id: build_jd_vector(jd_data["skills"])
    for jd_id, jd_data in job_descriptions.items()
}

# Cosine(A, B) = (A · B) / (|A| x |B|)
def cosine_similarity(tfidf_vec, jd_vec):
    dot_product = sum(tfidf_vec[s] * jd_vec[s] for s in vocabulary)
    mag_a = math.sqrt(sum(tfidf_vec[s] ** 2 for s in vocabulary))
    mag_b = math.sqrt(sum(jd_vec[s] ** 2 for s in vocabulary))
    if mag_a == 0 or mag_b == 0:
        return 0.0
    return dot_product / (mag_a * mag_b)

In [15]:
results = {}
for jd_id in job_descriptions:
    scores = [
        (name, cosine_similarity(tfidf_vectors[name], jd_vectors[jd_id]))
        for name in resumes_normalized
    ]
    # Sort: highest score first, alphabetical for ties
    scores.sort(key=lambda x: (-x[1], x[0]))
    results[jd_id] = scores[:3]

print("=" * 50)
print("RESUME MATCHING ENGINE — FINAL RESULTS")
print("=" * 50)
for jd_id, top3 in results.items():
    jd = job_descriptions[jd_id]
    print(f"\n{jd_id} — {jd['company']} ({jd['role']})")
    print(", ".join(f"{name}({score:.2f})" for name, score in top3))

print("\n\n--- Normalized Skills (for verification) ---")
for name, skills in resumes_normalized.items():
    print(f"{name} ({len(skills)} skills): {skills}")

RESUME MATCHING ENGINE — FINAL RESULTS

JD-1 — Kakao (ML Engineer)
Sneha Patel(0.57), Karan Mehta(0.53), Arjun Sharma(0.40)

JD-2 — Naver (Backend Engineer)
Rahul Gupta(0.81), Ananya Krishnan(0.28), Deepika Rao(0.19)

JD-3 — Line (Frontend Engineer)
Aditya Kumar(0.67), Priya Nair(0.58), Ananya Krishnan(0.35)


--- Normalized Skills (for verification) ---
Arjun Sharma (6 skills): ['python', 'machine_learning', 'sql', 'pandas', 'numpy', 'deep_learning']
Priya Nair (6 skills): ['javascript', 'react', 'nodejs', 'mongodb', 'rest_api', 'html_css']
Rahul Gupta (6 skills): ['java', 'spring_boot', 'mysql', 'microservices', 'docker', 'kubernetes']
Sneha Patel (6 skills): ['python', 'tensorflow', 'keras', 'nlp', 'bert', 'data_visualization']
Vikram Singh (5 skills): ['cpp', 'algorithms', 'data_structures', 'competitive_programming', 'python']
Ananya Krishnan (7 skills): ['javascript', 'vue', 'python', 'flask', 'postgresql', 'aws', 'ci_cd']
Karan Mehta (6 skills): ['python', 'machine_learning', 'x